# plotly-upset Showcase

Five examples demonstrating all feature combinations of the `plotly-upset` library.

## Example 1 — Minimal UpSet Plot (Defaults)

Create a basic UpSet plot with synthetic binary data using all default settings:
intersection bars sorted by cardinality, dot matrix, and set-size bars.

In [ ]:
import numpy as np
import pandas as pd

from plotly_upset import UpSetAnnotation, UpSetPlot

# Synthetic binary data: 6 sets, 200 elements
np.random.seed(42)
n = 200
df = pd.DataFrame(
    {
        "Set A": np.random.choice([0, 1], n, p=[0.6, 0.4]),
        "Set B": np.random.choice([0, 1], n, p=[0.5, 0.5]),
        "Set C": np.random.choice([0, 1], n, p=[0.7, 0.3]),
        "Set D": np.random.choice([0, 1], n, p=[0.65, 0.35]),
        "Set E": np.random.choice([0, 1], n, p=[0.75, 0.25]),
        "Set F": np.random.choice([0, 1], n, p=[0.8, 0.2]),
    }
)

plot = UpSetPlot(df)
fig = plot.to_plotly()
fig.show()

## Example 2 — Per-Set Coloring + Sorting & Filtering

Same dataset, but with per-set colors applied to dots, edges, and set-size bars.
Intersection bars are colored by set membership (degree-1 bars get their set color).
Also sorted by degree, filtered to max degree 3 and minimum size 5.

In [ ]:
# Reuse df from Example 1
set_colors = {
    "Set A": "#E41A1C",
    "Set B": "#377EB8",
    "Set C": "#4DAF4A",
    "Set D": "#984EA3",
    "Set E": "#FF7F00",
    "Set F": "#A65628",
}

plot = UpSetPlot(
    df,
    set_colors=set_colors,
    color_intersections_by="set",
    sort_by="degree",
    sort_order="ascending",
    min_size=5,
    max_degree=3,
    exclude_empty=True,
    title="Gene Set Overlaps",
    subtitle="Per-set coloring, filtered to degree \u2264 3, minimum size 5",
)
fig = plot.to_plotly()
fig.show()

## Example 3 — Box + Violin Annotations with Per-Set Colors

Binary DataFrame with two extra numeric columns. Demonstrates box track
(auto-detected) and explicit violin track, with per-set colored dots and edges.

In [ ]:
np.random.seed(123)
n = 150
df3 = pd.DataFrame(
    {
        "Pathway A": np.random.choice([0, 1], n, p=[0.55, 0.45]),
        "Pathway B": np.random.choice([0, 1], n, p=[0.50, 0.50]),
        "Pathway C": np.random.choice([0, 1], n, p=[0.65, 0.35]),
        "Pathway D": np.random.choice([0, 1], n, p=[0.60, 0.40]),
        "score": np.random.randn(n) * 2 + 5,
        "expression": np.random.lognormal(1, 0.8, n),
    }
)

anno = UpSetAnnotation(
    data=df3,
    score="score",  # auto-detected as box (numeric)
    expression={"column": "expression", "type": "violin", "color": "#E45756"},
)

plot = UpSetPlot(
    df3,
    set_columns=["Pathway A", "Pathway B", "Pathway C", "Pathway D"],
    set_colors=["#1B9E77", "#D95F02", "#7570B3", "#E7298A"],
    annotation=anno,
    title="Pathway Analysis",
    subtitle="Score (box) and Expression (violin) per intersection, with set colors",
    min_size=2,
)
fig = plot.to_plotly()
fig.show()

## Example 4 — Bar + Scatter + Categorical Annotations with Value Labels

Demonstrates aggregated bar track (median), scatter track (mean),
categorical track rendered as vertical stacked bars, and `show_values=True`
for intersection count labels.

In [ ]:
np.random.seed(456)
n = 180
df4 = pd.DataFrame(
    {
        "Module 1": np.random.choice([0, 1], n, p=[0.55, 0.45]),
        "Module 2": np.random.choice([0, 1], n, p=[0.50, 0.50]),
        "Module 3": np.random.choice([0, 1], n, p=[0.60, 0.40]),
        "Module 4": np.random.choice([0, 1], n, p=[0.70, 0.30]),
        "Module 5": np.random.choice([0, 1], n, p=[0.65, 0.35]),
        "quality": np.random.uniform(0, 100, n),
        "response_time": np.random.exponential(50, n),
        "category": np.random.choice(["High", "Medium", "Low"], n, p=[0.3, 0.5, 0.2]),
    }
)

anno = UpSetAnnotation(
    data=df4,
    quality={"column": "quality", "type": "bar", "agg": "median", "color": "#54A24B"},
    response_time={"column": "response_time", "type": "scatter", "agg": "mean", "color": "#FF9D00"},
    category="category",  # renders as vertical stacked bars
)

plot = UpSetPlot(
    df4,
    set_columns=["Module 1", "Module 2", "Module 3", "Module 4", "Module 5"],
    annotation=anno,
    title="Module Overlap Analysis",
    subtitle="Quality (bar), Response Time (scatter), Category (stacked bars), with value labels",
    min_size=3,
    sort_by="cardinality",
    show_values=True,
)
fig = plot.to_plotly()
fig.show()

## Example 5 — `from_sets()` with Per-Set Colors + Stacked Bar Annotation

Build from a `dict[str, set]` with separate metadata DataFrame for annotations.
Combines: per-set coloring, set-colored intersection bars, stacked bar annotation,
custom title/subtitle, hidden set-size bars, and custom dot size.

In [ ]:
np.random.seed(789)

# Define named sets (e.g., gene lists)
gene_sets = {
    "Apoptosis": set(range(0, 40)),
    "Cell Cycle": set(range(20, 55)),
    "DNA Repair": set(range(35, 65)),
    "Metabolism": set(range(50, 80)),
}

# Build metadata for all elements
all_elements = sorted(set().union(*gene_sets.values()))
metadata = pd.DataFrame(
    {
        "score_A": np.random.uniform(0, 10, len(all_elements)),
        "score_B": np.random.uniform(0, 8, len(all_elements)),
        "score_C": np.random.uniform(0, 5, len(all_elements)),
    },
    index=all_elements,
)

# Stacked bar annotation across multiple score columns
anno = UpSetAnnotation(
    data=metadata,
    scores={
        "column": "score_A",
        "type": "stacked_bar",
        "stack_columns": ["score_A", "score_B", "score_C"],
        "agg": "mean",
    },
)

plot = UpSetPlot.from_sets(
    gene_sets,
    data=metadata,
    annotation=anno,
    set_colors={
        "Apoptosis": "#E41A1C",
        "Cell Cycle": "#377EB8",
        "DNA Repair": "#4DAF4A",
        "Metabolism": "#984EA3",
    },
    color_intersections_by="set",
    show_values=True,
    title="Gene Set Intersection Analysis",
    subtitle="Per-set colors, stacked bar annotation with mean scores",
    show_set_sizes=False,
    dot_size=16,
    sort_by="cardinality",
    min_size=1,
    width=1000,
    height=600,
)
fig = plot.to_plotly()
fig.show()